# 10 - Model Calibration

Assess the calibration of predicted probabilities for all three classifiers
using out-of-fold predictions on GSE96058 combined features.

Well-calibrated models produce predicted probabilities that match observed
event rates (e.g., among patients predicted ~30% risk, ~30% actually have events).

**Requires**: Downloaded GSE96058 expression data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.base import clone

from src.data_loader import load_clinical_data, load_gse96058_expression
from src.preprocessing import zscore_normalize, encode_clinical_features, filter_outcome
from src.features import compute_pathway_scores, add_ratio_features, build_feature_matrix
from src.models import get_classifiers

## 1. Prepare Data

In [ ]:
gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
gse_clin = filter_outcome(gse_clin)
gse_exp = load_gse96058_expression('../data/raw/GSE96058_gene_expression.csv')

common = list(set(gse_clin['sample_id']) & set(gse_exp['sample_id']))
gse_clin = gse_clin[gse_clin['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)
gse_exp = gse_exp[gse_exp['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)

gse_exp_norm = zscore_normalize(gse_exp)
pathway_features = compute_pathway_scores(gse_exp_norm)
pathway_features = add_ratio_features(pathway_features)
clinical_features = encode_clinical_features(gse_clin)
X = build_feature_matrix(pathway_features, clinical_features)
y = gse_clin['high_risk'].values

print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")

## 2. Collect Out-of-Fold Predictions

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_arr = np.array(X)
classifiers = get_classifiers()

# Store out-of-fold predictions for each model
oof_predictions = {name: np.zeros(len(y)) for name in classifiers}

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_arr, y)):
    print(f"Fold {fold_idx + 1}/5...")
    X_train, X_test = X_arr[train_idx], X_arr[test_idx]
    y_train = y[train_idx]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    for name, clf_template in classifiers.items():
        clf = clone(clf_template)
        clf.fit(X_train_s, y_train)
        oof_predictions[name][test_idx] = clf.predict_proba(X_test_s)[:, 1]

print("\nCollected out-of-fold predictions for all models.")

## 3. Calibration Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

colors = {'Elastic Net': '#1f77b4', 'Random Forest': '#ff7f0e', 'Gradient Boosting': '#2ca02c'}

# Perfect calibration reference
ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated', alpha=0.7)

for name, y_prob in oof_predictions.items():
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y, y_prob, n_bins=10, strategy='uniform'
    )
    ax.plot(mean_predicted_value, fraction_of_positives, 's-',
            label=name, color=colors[name], linewidth=2, markersize=7)

ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Fraction of Positives', fontsize=12)
ax.set_title('Calibration Curves - GSE96058 Combined Features', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('../figures/fig_calibration.png', dpi=300, bbox_inches='tight')
print("Saved figures/fig_calibration.png")
plt.show()